## Question 5 - *Percentage of jobs/tasks killed or evicted: is it important?*

**Metric.**
For jobs and tasks separately:
- `% killed = (# distinct entities with ≥1 KILL) / (total # distinct entities)`
- `% evicted = (# distinct entities with ≥1 EVICT) / (total # distinct entities)`
- `% killed-or-evicted = (# distinct entities with ≥1 {KILL or EVICT}) / total`

We count each entity once even if multiple events occur.

**Method (Spark RDD).**
1. Parse job_events → `(job_id, event_type)` and count distinct jobs.
2. Parse task_events → `((job_id, task_index), event_type)` and count distinct tasks.
3. Count distinct jobs/tasks that appear with `event_type in {EVICT=2, KILL=5}`.
4. Compute percentages and validate event availability in the subset.

In [13]:
import sys
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F
import time
from pyspark.storagelevel import StorageLevel
import pandas as pd

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

PATH_MACHINE_EVENTS = "./data/machine_events/*.csv.gz"
PATH_JOB_EVENTS  = "./data/job_events/*.csv.gz"
PATH_TASK_EVENTS = "./data/task_events/*.csv.gz"
PATH_TASK_USAGE = "./data/task_usage/*.csv.gz"

# ----------------------------
# event_type codes
# ----------------------------
EVICT = 2
KILL  = 5

# ----------------------------
# Parsing job events
# ----------------------------

def parse_job_event(line):
    p = line.split(",")
    if len(p) < 4:
        return None
    ts = to_long(p[0])
    job_id = to_long(p[2])
    et = to_int(p[3])
    if ts is None or job_id is None or et is None:
        return None
    return (job_id, et)

def parse_task_event(line):
    p = line.split(",")
    if len(p) < 6:
        return None
    ts = to_long(p[0])
    job_id = to_long(p[2])
    task_index = to_int(p[3])
    et = to_int(p[5])
    if ts is None or job_id is None or task_index is None or et is None:
        return None
    return ((job_id, task_index), et)

# ----------------------------
# Parsing utilities
# ----------------------------
def to_int(x):
    return int(x) if x not in (None, "", "NULL") else None

def to_long(x):
    return int(x) if x not in (None, "", "NULL") else None

def to_float(x):
    return float(x) if x not in (None, "", "NULL") else None

In [14]:
# ----------------------------
# Load
# ----------------------------
job_events = sc.textFile(PATH_JOB_EVENTS).map(parse_job_event).filter(lambda x: x is not None).cache()
task_events = sc.textFile(PATH_TASK_EVENTS).map(parse_task_event).filter(lambda x: x is not None).cache()

In [15]:
# ----------------------------
# Totals
# ----------------------------
total_jobs = job_events.map(lambda x: x[0]).distinct().count()
total_tasks = task_events.map(lambda x: x[0]).distinct().count()

print("Total distinct jobs :", total_jobs)
print("Total distinct tasks:", total_tasks)

Total distinct jobs : 6551
Total distinct tasks: 246415


In [16]:
# ----------------------------
# Jobs: killed/evicted
# ----------------------------
jobs_evicted = job_events.filter(lambda x: x[1] == EVICT).map(lambda x: x[0]).distinct().count()
jobs_killed  = job_events.filter(lambda x: x[1] == KILL ).map(lambda x: x[0]).distinct().count()

jobs_killed_or_evicted = (
    job_events
    .filter(lambda x: x[1] in (EVICT, KILL))
    .map(lambda x: x[0])
    .distinct()
    .count()
)

# ----------------------------
# Tasks: killed/evicted
# ----------------------------
tasks_evicted = task_events.filter(lambda x: x[1] == EVICT).map(lambda x: x[0]).distinct().count()
tasks_killed  = task_events.filter(lambda x: x[1] == KILL ).map(lambda x: x[0]).distinct().count()

tasks_killed_or_evicted = (
    task_events
    .filter(lambda x: x[1] in (EVICT, KILL))
    .map(lambda x: x[0])
    .distinct()
    .count()
)

# ----------------------------
# Percentages
# ----------------------------
def pct(part, whole):
    return (100.0 * part / whole) if whole else 0.0

In [17]:
print("\n--- JOBS ---")
print(f"Evicted: {jobs_evicted} ({pct(jobs_evicted, total_jobs):.3f}%)")
print(f"Killed : {jobs_killed} ({pct(jobs_killed, total_jobs):.3f}%)")
print(f"Killed or Evicted: {jobs_killed_or_evicted} ({pct(jobs_killed_or_evicted, total_jobs):.3f}%)")

print("\n--- TASKS ---")
print(f"Evicted: {tasks_evicted} ({pct(tasks_evicted, total_tasks):.3f}%)")
print(f"Killed : {tasks_killed} ({pct(tasks_killed, total_tasks):.3f}%)")
print(f"Killed or Evicted: {tasks_killed_or_evicted} ({pct(tasks_killed_or_evicted, total_tasks):.3f}%)")


--- JOBS ---
Evicted: 0 (0.000%)
Killed : 2465 (37.628%)
Killed or Evicted: 2465 (37.628%)

--- TASKS ---
Evicted: 24090 (9.776%)
Killed : 59078 (23.975%)
Killed or Evicted: 77862 (31.598%)


In [18]:
# Validatio check : What job event_types exist in your subset?
job_event_type_counts = (
    job_events
    .map(lambda x: (x[1], 1))
    .reduceByKey(lambda a,b: a+b)
    .collect()
)

for et, cnt in sorted(job_event_type_counts):
    print(et, cnt)

0 6255
1 6254
3 85
4 3671
5 2465
6 1


In [19]:
# Count how many eviction events per task
task_evict_counts = (
    task_events
    .filter(lambda x: x[1] == EVICT)
    .map(lambda x: (x[0], 1))
    .reduceByKey(lambda a,b: a+b)
)

# Simple stats: min / mean / max of eviction counts per evicted task
evict_vals = task_evict_counts.map(lambda x: x[1])
print("Evicted tasks:", task_evict_counts.count())
print("Evictions per evicted task -> min:", evict_vals.min(), "mean:", evict_vals.mean(), "max:", evict_vals.max())

Evicted tasks: 24090
Evictions per evicted task -> min: 1 mean: 1.7803237858032397 max: 16


**Results (subset).**
- Total distinct jobs: **6,551**
- Total distinct tasks: **246,415**

| Entity | Evicted | Killed | Killed or Evicted |
|---|---:|---:|---:|
| Jobs | 0 (0.000%) | 2,465 (37.628%) | 2,465 (37.628%) |
| Tasks | 24,090 (9.776%) | 59,078 (23.975%) | 77,862 (31.598%) |

Validation: job event types observed in the subset are `{0,1,3,4,5,6}` (no `event_type=2`), so **job evictions cannot be measured here**.

Extra insight (tasks only):
- #evicted tasks: **24,090**
- Evictions per evicted task: min=1, mean=1.78, max=16

**Interpretation.**
- At the task level, **31.6%** killed-or-evicted is high (≈ 1 task out of 3), suggesting that interruptions are part of “normal” cluster dynamics.
- Evictions affect ~9.8% of tasks and can repeat, consistent with preemption/rescheduling.
- Job-level evictions are absent because our job_events subset does not contain EVICT events, not because they never occur in the full trace.